# Lakebase 101 — Post-Deploy

Run this **after** `databricks bundle deploy`.

1. Grants the app's service principal `CAN_RUN` on the synced-table pipelines (the **Sync Now** button)
2. Grants the app SP **Unity Catalog** read access — the speed test & analytics panes query the gold/source tables via the SQL Warehouse *as the SP*
3. Grants **Postgres** schema access and creates + seeds the OLTP tables (`orders`, `inventory`) the app reads and writes


In [0]:
CATALOG = "lakebase_101_catalog"
APP_NAME = "lakebase-101-app"

In [0]:
"""Grant the app's service principal CAN_MANAGE_RUN on synced-table pipelines.
This allows the 'Sync Now' button in the app to trigger on-demand refreshes."""
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import PipelineAccessControlRequest, PipelinePermissionLevel

APP_NAME = "lakebase-101-app"

w = WorkspaceClient()

# Wait briefly for the app to be registered after deploy
for attempt in range(5):
    try:
        app = w.apps.get(APP_NAME)
        sp_name = app.service_principal_client_id
        print(f"App SP (client_id): {sp_name}")
        print(f"App SP (display):   {app.service_principal_name}")
        break
    except Exception:
        if attempt < 4:
            time.sleep(5)
        else:
            raise RuntimeError(f"App '{APP_NAME}' not found after deploy.")

# Find synced-table pipelines and grant permissions
granted = 0
for p in w.pipelines.list_pipelines(filter=f"name LIKE '%{CATALOG}%'"):
    try:
        w.pipelines.update_permissions(
            pipeline_id=p.pipeline_id,
            access_control_list=[
                PipelineAccessControlRequest(
                    service_principal_name=sp_name,
                    permission_level=PipelinePermissionLevel.CAN_RUN
                )
            ]
        )
        granted += 1
        print(f"  ✅ {p.pipeline_id} | {p.name}")
    except Exception as e:
        print(f"  ⚠️  {p.pipeline_id}: {e}")

print(f"\n✅ Granted CAN_MANAGE_RUN on {granted} pipeline(s) to {sp_name}")

App SP (client_id): 4713da01-550f-40fa-9645-4c11a01beb29
App SP (display):   app-1s0sa2 lakebase-101-app
  ✅ 0f01e463-2da2-4b3f-8afe-15c800525c99 | Synced table: lakebase_101_catalog.lakebase_101_schema.customers_directory_synced HStSjQ
  ✅ 7c463bee-33b9-4030-9fc1-a4f9ced3b6db | Synced table: lakebase_101_catalog.lakebase_101_schema.sales_events_synced xZh477
  ✅ 8056f305-9c07-43d6-b8b3-222ba191dfd1 | Synced table: lakebase_101_catalog.lakebase_101_schema.customer_360_synced PaGOmL

✅ Granted CAN_MANAGE_RUN on 3 pipeline(s) to 4713da01-550f-40fa-9645-4c11a01beb29


In [0]:
"""Grant the app's service principal Unity Catalog read access.
The speed test (/api/speed) and analytics showdown (/api/aggregate) query the gold
and source tables via the SQL Warehouse *as the app SP*. Without USE_CATALOG /
USE_SCHEMA / SELECT those warehouse queries fail with a permission error."""
from databricks.sdk.service.catalog import PermissionsChange, Privilege, SecurableType

SCHEMA = "lakebase_101_schema"
role = w.apps.get(APP_NAME).service_principal_client_id  # same client_id the app uses at runtime

w.grants.update(
    securable_type=SecurableType.CATALOG, full_name=CATALOG,
    changes=[PermissionsChange(principal=role, add=[Privilege.USE_CATALOG])],
)
w.grants.update(
    securable_type=SecurableType.SCHEMA, full_name=f"{CATALOG}.{SCHEMA}",
    changes=[PermissionsChange(principal=role, add=[Privilege.USE_SCHEMA, Privilege.SELECT])],
)
print(f"✅ UC grants (USE_CATALOG / USE_SCHEMA / SELECT) applied to {role}")

In [0]:
"""Postgres setup for the app SP:
  1. USAGE + SELECT on the synced-table schema (so the app reads the reverse-ETL tables)
  2. Create the OLTP tables the app owns (orders, inventory) + grant the SP read/write
  3. Seed inventory from the products Delta table
Derives the Postgres role from the app's service_principal_client_id (no hardcoded UUIDs)."""
import psycopg2

PROJECT_ID  = "lakebase-101-demo"
BRANCH_ID   = "production"
ENDPOINT_ID = "primary"
DATABASE    = "lakebase_101_db"
SCHEMA      = "lakebase_101_schema"

role = w.apps.get(APP_NAME).service_principal_client_id
print(f"App SP role (client_id): {role}")

# Endpoint host + a Lakebase-scoped JWT (we connect as the deploying user, who owns the schema)
endpoint_name = f"projects/{PROJECT_ID}/branches/{BRANCH_ID}/endpoints/{ENDPOINT_ID}"
ep = w.postgres.get_endpoint(name=endpoint_name)
host = ep.status.hosts.host
print(f"Lakebase host: {host}")

token = w.postgres.generate_database_credential(endpoint=endpoint_name).token
user = w.current_user.me().user_name

conn = psycopg2.connect(
    host=host, port=5432, dbname=DATABASE,
    user=user, password=token, sslmode="require", connect_timeout=10,
)
conn.autocommit = True

with conn.cursor() as cur:
    # 1. Read access to the synced-table schema (persists across syncs; DEFAULT PRIVILEGES
    #    covers tables a SNAPSHOT sync re-creates)
    cur.execute(f'GRANT USAGE ON SCHEMA {SCHEMA} TO "{role}";')
    cur.execute(f'GRANT SELECT ON ALL TABLES IN SCHEMA {SCHEMA} TO "{role}";')
    cur.execute(f'ALTER DEFAULT PRIVILEGES IN SCHEMA {SCHEMA} GRANT SELECT ON TABLES TO "{role}";')
    print(f"✅ Synced-table grants on schema {SCHEMA}")

    # 2. OLTP tables the app owns (place-order writes; stats tiles + customer-360 read).
    #    In public because the app queries them unqualified (search_path = "$user", public).
    #    GENERATED ALWAYS AS IDENTITY -> no sequence USAGE grant needed for the SP.
    cur.execute("""
        CREATE TABLE IF NOT EXISTS public.inventory (
          product_id    int PRIMARY KEY,
          product_name  text NOT NULL,
          category      text,
          price         numeric(10,2) NOT NULL,
          stock_on_hand int NOT NULL DEFAULT 0,
          updated_at    timestamptz NOT NULL DEFAULT now()
        );""")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS public.orders (
          order_id     bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
          customer_id  int  NOT NULL,
          product_id   int  NOT NULL,
          product_name text,
          quantity     int  NOT NULL,
          unit_price   numeric(10,2) NOT NULL,
          total        numeric(12,2) NOT NULL,
          status       text NOT NULL DEFAULT 'confirmed',
          created_at   timestamptz NOT NULL DEFAULT now()
        );""")
    cur.execute(f'GRANT USAGE ON SCHEMA public TO "{role}";')
    for t in ("inventory", "orders"):
        cur.execute(f'GRANT SELECT, INSERT, UPDATE, DELETE ON public.{t} TO "{role}";')
    print("✅ Created public.inventory / public.orders and granted the app SP read/write")

# 3. Seed inventory from the products Delta table (idempotent)
products = (spark.table(f"{CATALOG}.{SCHEMA}.products")
                 .select("product_id", "product_name", "category", "price").collect())
with conn.cursor() as cur:
    for p in products:
        cur.execute(
            """INSERT INTO public.inventory (product_id, product_name, category, price, stock_on_hand)
               VALUES (%s, %s, %s, %s, %s)
               ON CONFLICT (product_id) DO NOTHING""",
            (p.product_id, p.product_name, p.category, float(p.price), 500),
        )
print(f"✅ Seeded inventory: {len(products)} products (stock_on_hand=500 each)")

conn.close()
print(f"\n✅ Postgres setup complete for role {role}")